# P9 — To-nivaa-strukturen: empirisk bekreftelse

**Teori (K19):**

```
Indre nivaa (Lovgiveren, P7):
    K        = Sigma H_spektral(W_l)
    C0_indre = rho x K

Ytre nivaa (Tolken, P1):
    C0_ytre  = K x log2(hidden_dim)

Broen:
    Lambda   = log2(hidden_dim) / rho
    C0_ytre  = C0_indre x Lambda
```

**Testbar prediksjon for gpt-neo-1.3B:**
```
K = 1555.77  (malt i P7/P8)
C0_ytre = 1555.77 x log2(2048) = 1555.77 x 11.0 = 17 113 bits
```

**Kjør alle celler fra topp til bunn.**

In [ ]:
# CELLE 1: Installer
!pip install -q transformers torch accelerate

In [ ]:
# CELLE 2: LIMFilter
import math, time, json
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

class LIMFilter:
    def __init__(self):
        self.gamma  = 0.5772156649
        self.delta  = 4.6692016091
        self.zeta3  = 1.2020569032
        self.rho    = self.gamma / (self.delta - 4)   # 0.8625437492
        self.tau_lo = math.exp(-self.gamma)            # 0.5615
        self.tau_hi = 1.0 / self.zeta3                # 0.8319
        self.C0_VALO = 4495.27                        # VALO OS v1.6 konstant

    def spectral_entropy(self, W):
        if W.numel() == 0 or W.ndim < 2:
            return 0.0
        try:
            Wf = W.float()
            md = min(Wf.shape)
            if md > 2048:
                _, S, _ = torch.svd_lowrank(Wf, q=min(512, md))
            else:
                _, S, _ = torch.linalg.svd(Wf, full_matrices=False)
            s = S.detach().cpu().numpy()
            s = s[s > 1e-10]
            if len(s) == 0:
                return 0.0
            p = s / s.sum()
            return float(-(p * np.log2(p + 1e-10)).sum())
        except Exception:
            return 0.0

    def compute_K(self, model):
        K, rows = 0.0, []
        for name, p in model.named_parameters():
            if p.ndim >= 2 and p.shape[0] > 1 and p.shape[1] > 1:
                H = self.spectral_entropy(p.detach())
                if H > 0:
                    K += H
                    rows.append((H, name, list(p.shape)))
        rows.sort(reverse=True)
        return K, rows

    def tau_layer(self, hidden_state):
        """Shannon-entropi av egenverdier til kovariansmatrisen (bits)."""
        bs, sl, hd = hidden_state.shape
        x = hidden_state.reshape(-1, hd).float()
        x = x - x.mean(0)
        L = (x.T @ x) / sl + 1e-6 * torch.eye(hd, device=x.device)
        ev = torch.linalg.eigvalsh(L)
        ev = ev[ev > 0]
        p  = ev / ev.sum()
        p  = p[p > 1e-10]
        return float(-(p * torch.log2(p)).sum())

lim = LIMFilter()
print(f"rho        = {lim.rho:.10f}")
print(f"C0_VALO    = {lim.C0_VALO}")
print(f"Goldilocks = [{lim.tau_lo:.4f}, {lim.tau_hi:.4f}]")
print("LIMFilter klar.")

In [ ]:
# CELLE 3: Maalingsfunksjon
def measure_model(model_name, num_samples=5):
    print(f"\n{'='*60}")
    print(f"MODELL: {model_name}")
    print('='*60)

    tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    mdl = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    mdl.eval()
    cfg = mdl.config
    hd  = cfg.hidden_size
    n_lag = cfg.num_hidden_layers
    log2_hd = math.log2(hd)
    print(f"  hidden_dim = {hd}, lag = {n_lag}, log2(hd) = {log2_hd:.4f}")

    # --- K fra vektmatriser (Lovgiverens rike) ---
    print("\n[1] K fra vektmatriser (Lovgiverens rike)...")
    t0 = time.time()
    K, rows = lim.compute_K(mdl)
    n_mat = len(rows)
    C0_indre = lim.rho * K
    C0_ytre  = K * log2_hd
    lam      = log2_hd / lim.rho
    print(f"  K_spektral           = {K:.4f}")
    print(f"  C0_indre (rho x K)   = {C0_indre:.4f}")
    print(f"  C0_ytre (K x log2hd) = {C0_ytre:.4f}")
    print(f"  Lambda (log2hd/rho)  = {lam:.4f}")
    print(f"  Matriser: {n_mat}, tid: {time.time()-t0:.1f}s")

    # --- Sammenlign med VALO-konstanten (kun for GPT-2) ---
    if hd == 768:
        avvik = abs(C0_ytre - lim.C0_VALO) / lim.C0_VALO * 100
        print(f"\n  VALO-konstant kalibrering:")
        print(f"  C0_ytre_pred  = {C0_ytre:.2f}")
        print(f"  C0_VALO       = {lim.C0_VALO}")
        print(f"  Avvik         = {avvik:.2f}%")

    # --- tau per lag (Tolkens rike) ---
    print(f"\n[2] tau per lag (Tolkens rike, {num_samples} samples)...")
    seed = "The coherence of a system is determined by its ability to maintain identity through constrained boundaries."
    device = next(mdl.parameters()).device

    tau_last_list, tau_sum_list = [], []
    for i in range(num_samples):
        inp = tok((seed * 4)[:256], return_tensors='pt', truncation=True, max_length=256)
        inp = {k: v.to(device) for k, v in inp.items()}
        with torch.no_grad():
            out = mdl(**inp, output_hidden_states=True)
        tau_last = lim.tau_layer(out.hidden_states[-1])
        tau_sum  = sum(lim.tau_layer(h) for h in out.hidden_states)
        tau_last_list.append(tau_last)
        tau_sum_list.append(tau_sum)

    tl = float(np.median(tau_last_list))
    ts = float(np.median(tau_sum_list))

    print(f"  tau_last (siste lag)    = {tl:.4f} bits")
    print(f"  tau_sum  (sum alle lag) = {ts:.4f} bits")
    print(f"  tau_sum / C0_ytre       = {ts/C0_ytre:.6f}")
    print(f"  C0_ytre / tau_sum       = {C0_ytre/ts:.4f}  [forstorrelsesforholdet]")

    del mdl
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "model": model_name,
        "hidden_dim": hd,
        "num_layers": n_lag,
        "num_matrices": n_mat,
        "log2_hd": log2_hd,
        "K": K,
        "C0_indre": C0_indre,
        "C0_ytre": C0_ytre,
        "lambda": lam,
        "tau_last": tl,
        "tau_sum": ts,
    }

print("Funksjon klar.")

In [ ]:
# CELLE 4: GPT-2 — kalibreringssjekk mot VALO-konstanten
# Forventet: K=463, C0_ytre = 463 x 9.585 = 4438 ~= 4495 (1.3% avvik)
r_gpt2 = measure_model("gpt2")

In [ ]:
# CELLE 5: gpt-neo-1.3B — prediksjon
# Forventet: K=1555.77, C0_ytre = 1555.77 x 11.0 = 17 113 bits
r_neo = measure_model("EleutherAI/gpt-neo-1.3B")

In [ ]:
# CELLE 6: To-nivaa-verifikasjon
resultater = [r_gpt2, r_neo]

print("="*70)
print("TO-NIVAA STRUKTUR — VERIFIKASJON")
print("="*70)
print(f"{'Modell':<28} {'K':>8} {'C0_indre':>10} {'C0_ytre':>10} {'Lambda':>8}")
print(f"{'-'*65}")
for r in resultater:
    print(f"{r['model']:<28} {r['K']:>8.1f} {r['C0_indre']:>10.1f} {r['C0_ytre']:>10.1f} {r['lambda']:>8.4f}")

print("\n--- Forholdstall ---")
k_ratio     = r_neo['K'] / r_gpt2['K']
c0i_ratio   = r_neo['C0_indre'] / r_gpt2['C0_indre']
c0y_ratio   = r_neo['C0_ytre'] / r_gpt2['C0_ytre']
log2_ratio  = r_neo['log2_hd'] / r_gpt2['log2_hd']

print(f"  K_neo / K_gpt2            = {k_ratio:.4f}x")
print(f"  C0_indre_neo / C0_i_gpt2  = {c0i_ratio:.4f}x   (skal = K-ratio)")
print(f"  C0_ytre_neo  / C0_y_gpt2  = {c0y_ratio:.4f}x   (skal = K-ratio x log2-ratio)")
print(f"  log2(2048) / log2(768)    = {log2_ratio:.4f}x")
print(f"  K-ratio x log2-ratio      = {k_ratio * log2_ratio:.4f}x   (skal = C0_ytre-ratio)")

print("\n--- VALO-kalibrering ---")
print(f"  C0_VALO                   = {lim.C0_VALO}")
print(f"  C0_ytre_GPT2              = {r_gpt2['C0_ytre']:.2f}")
print(f"  Avvik                     = {abs(r_gpt2['C0_ytre'] - lim.C0_VALO)/lim.C0_VALO*100:.2f}%")

print("\n--- Prediksjon vs malt (neo-1.3B) ---")
pred_neo = r_gpt2['K'] / r_gpt2['log2_hd'] * r_neo['log2_hd']  # K_neo_pred fra GPT-2
c0y_pred = r_neo['K'] * r_neo['log2_hd']
print(f"  K_neo (P8 malt):          = 1555.77")
print(f"  K_neo (re-malt her):      = {r_neo['K']:.2f}")
print(f"  C0_ytre_neo_pred (K19):   = {1555.77 * 11.0:.1f}")
print(f"  C0_ytre_neo (re-beregnet) = {c0y_pred:.1f}")

print("\nTofoo. Phi")

with open('p9_resultater.json', 'w') as f:
    json.dump({'gpt2': r_gpt2, 'neo': r_neo,
               'k_ratio': k_ratio, 'c0y_ratio': c0y_ratio,
               'log2_ratio': log2_ratio}, f, indent=2)
print("Lagret: p9_resultater.json")